# Gradient Boosting Survival Analysis

Trains a `GradientBoostingSurvivalAnalysis` model on `mm_nopgs` and `mm_pgs_bin` with grid search over key hyperparameters.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import RandomizedSearchCV
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.util import Surv

## Load data

In [2]:
for p in ['cleaned_data/mm_pgs_score.csv', 'cleaned_data/mm_nopgs.csv']:
    assert Path(p).exists(), f'{p} not found — run clean_data.ipynb first.'

df_pgs   = pd.read_csv('cleaned_data/mm_pgs_bin.csv', index_col='ID')
df_nopgs = pd.read_csv('cleaned_data/mm_nopgs.csv',   index_col='ID')

print(f'PGS dataset:    {len(df_pgs)} rows,   columns: {list(df_pgs.columns)}')
print(f'No-PGS dataset: {len(df_nopgs)} rows, columns: {list(df_nopgs.columns)}')

PGS dataset:    2000 rows,   columns: ['ancestry', 'age', 'm_spike', 'sflc_ratio', 'creatinine', 'pgs_score', 'status', 'time_years']
No-PGS dataset: 2000 rows, columns: ['ancestry', 'age', 'm_spike', 'sflc_ratio', 'creatinine', 'status', 'time_years']


## Grid search

In [3]:
features_nopgs = ['age', 'm_spike', 'sflc_ratio', 'creatinine']
features_pgs   = ['age', 'm_spike', 'sflc_ratio', 'creatinine', 'pgs_bin']

y_nopgs = Surv.from_dataframe('status', 'time_years', df_nopgs)
y_pgs   = Surv.from_dataframe('status', 'time_years', df_pgs)

param_dist = {
    'n_estimators':  [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth':     [2, 3, 4],
}

gbm = GradientBoostingSurvivalAnalysis(random_state=42)

gs_nopgs = RandomizedSearchCV(gbm, param_dist, n_iter=10, cv=5, refit=False, n_jobs=5, random_state=42)
gs_nopgs.fit(df_nopgs[features_nopgs], y_nopgs)

gs_pgs = RandomizedSearchCV(gbm, param_dist, n_iter=10, cv=5, refit=False, n_jobs=5, random_state=42)
gs_pgs.fit(df_pgs[features_pgs], y_pgs)

cols = ['param_n_estimators', 'param_learning_rate', 'param_max_depth', 'mean_test_score', 'std_test_score']

results_nopgs = pd.DataFrame(gs_nopgs.cv_results_)[cols].sort_values('mean_test_score', ascending=False)
results_pgs   = pd.DataFrame(gs_pgs.cv_results_)[cols].sort_values('mean_test_score', ascending=False)

print('mm_nopgs:')
print(results_nopgs.to_string(index=False))
print()
print('mm_pgs_bin:')
print(results_pgs.to_string(index=False))

mm_nopgs:
 param_n_estimators  param_learning_rate  param_max_depth  mean_test_score  std_test_score
                200                 0.10                2         0.690237        0.015582
                 50                 0.10                3         0.684631        0.017525
                 50                 0.10                2         0.684539        0.020830
                100                 0.10                3         0.684179        0.012162
                 50                 0.20                3         0.682523        0.012276
                 50                 0.05                2         0.678365        0.023618
                200                 0.05                4         0.677292        0.008435
                100                 0.10                4         0.676076        0.008961
                 50                 0.20                4         0.675512        0.007911
                200                 0.10                4         0.669283      